# DATA GEN

In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


# Generating questions with structured output

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# The instructions for the LLM:

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

Call the LLM for one document:

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
model="llama-3.3-70b-versatile"

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {
        "role": "system",
        "content": data_gen_instructions + """

Return ONLY valid JSON.

Format:

{
  "questions": [
    "...",
    "...",
    "...",
    "...",
    "..."
  ]
}

Do not return markdown.
Do not use ```json.
Do not add explanations.
"""
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

Call the model:

In [10]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    temperature=0
)

In [11]:
result = response.choices[0].message.content

print(result)

{
  "questions": [
    "What should I do if my homework answer doesn't match any of the given options?",
    "How can I troubleshoot when my answer doesn't match any of the options in the homework?",
    "Why doesn't my homework answer match any of the provided options, what are the common causes?",
    "I'm having trouble with my homework, my answer doesn't match any of the options, what should I check first?",
    "What are the possible reasons why my homework answer doesn't match any of the available options and how can I resolve the issue?"
  ]
}


In [12]:
questions = json.loads(result)

print(questions)

{'questions': ["What should I do if my homework answer doesn't match any of the given options?", "How can I troubleshoot when my answer doesn't match any of the options in the homework?", "Why doesn't my homework answer match any of the provided options, what are the common causes?", "I'm having trouble with my homework, my answer doesn't match any of the options, what should I check first?", "What are the possible reasons why my homework answer doesn't match any of the available options and how can I resolve the issue?"]}


In [13]:
print(questions["questions"])

["What should I do if my homework answer doesn't match any of the given options?", "How can I troubleshoot when my answer doesn't match any of the options in the homework?", "Why doesn't my homework answer match any of the provided options, what are the common causes?", "I'm having trouble with my homework, my answer doesn't match any of the options, what should I check first?", "What are the possible reasons why my homework answer doesn't match any of the available options and how can I resolve the issue?"]


In [14]:
for q in questions["questions"]:
    print(q)

What should I do if my homework answer doesn't match any of the given options?
How can I troubleshoot when my answer doesn't match any of the options in the homework?
Why doesn't my homework answer match any of the provided options, what are the common causes?
I'm having trouble with my homework, my answer doesn't match any of the options, what should I check first?
What are the possible reasons why my homework answer doesn't match any of the available options and how can I resolve the issue?


# Reusable utilities

In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    client,
    data_gen_instructions,
    user_prompt
)

print(result["questions"])

[{'id': 1, 'text': "What should I do if my homework answer doesn't match any of the given options"}, {'id': 2, 'text': "How can I troubleshoot my machine learning homework when the answer doesn't match any options"}, {'id': 3, 'text': 'Why does my machine learning homework answer not match any of the provided options'}, {'id': 4, 'text': 'What are the common causes for my machine learning homework answer not matching any options'}, {'id': 5, 'text': "How can I resolve the issue when my homework answer doesn't match any of the available options"}]


Tracking cost

In [17]:
usage.prompt_tokens
usage.completion_tokens
usage.total_tokens

507

In [19]:
print("Input tokens:", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Input tokens: 213
Output tokens: 145
Total tokens: 358


Calculate the cost of this call:

In [18]:
from evaluation_utils import calc_price

cost = calc_price(usage)

print(cost)

{'input_cost': 0.0, 'output_cost': 0.0, 'total_cost': 0.0}


Now convert these questions into ground truth records:

# Generating Ground Truth for All Documents

In [19]:
from evaluation_utils import llm_structured_retry

In [20]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,    
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out["questions"]:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [21]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [22]:
usages

[CompletionUsage(completion_tokens=145, prompt_tokens=213, total_tokens=358, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040970719, prompt_time=0.021458039, completion_time=0.575403273, total_time=0.596861312),
 CompletionUsage(completion_tokens=156, prompt_tokens=246, total_tokens=402, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040749509, prompt_time=0.049770518, completion_time=0.625071364, total_time=0.674841882),
 CompletionUsage(completion_tokens=145, prompt_tokens=319, total_tokens=464, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.042343213, prompt_time=0.016238541, completion_time=0.528020274, total_time=0.544258815),
 CompletionUsage(completion_tokens=166, prompt_tokens=383, total_tokens=549, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.042567559, prompt_time=0.019706883, completion_time=0.556563253, total_time=0.576270136),
 CompletionUsage(completion_tokens=140, 

In [23]:
ground_truth

[{'question': {'id': 1,
   'text': 'Is it too late to enroll in the course if I just found out about it now?'},
  'document': '74eb249bbf'},
 {'question': {'id': 2,
   'text': 'Can I still participate in the course if the start date has already passed?'},
  'document': '74eb249bbf'},
 {'question': {'id': 3,
   'text': 'How do I know if I can still join the course after the initial start date?'},
  'document': '74eb249bbf'},
 {'question': {'id': 4,
   'text': 'Will I be able to get a certificate if I join the course late?'},
  'document': '74eb249bbf'},
 {'question': {'id': 5,
   'text': 'What are the requirements to join the course after it has already started?'},
  'document': '74eb249bbf'},
 {'question': {'id': 1,
   'text': 'How do I know if my registration for the LLM Zoomcamp was successful?'},
  'document': '977bf7786c'},
 {'question': {'id': 2,
   'text': 'What happens after I sign up for the course, do I get some kind of confirmation?'},
  'document': '977bf7786c'},
 {'question

# Parallel processing

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kvxgvwrefrx8vsmnybefgp1p` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99944, Requested 363. Please try again in 4m25.248s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

## Create a dataframe so we can look at the records as a table and save them as a CSV file.

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)